In [2]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
# Q1. How many lesson pages
len(documents)

72

In [6]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)


In [7]:
# Q2. Indexing and searching
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [8]:
#!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

In [9]:
from rag_helper import RAGBase

class RAGLessons(RAGBase):
    def search(self, query, num_results=5):
        boost_dict = {'content': 3.0, 'filename': 0.5}
        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append('Lesson file name: ' + doc['filename'])
            lines.append('Content:')
            lines.append(doc['content'])
            lines.append('')

        return '\n'.join(lines).strip()
    
    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response
    
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response

In [10]:
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGLessons(
    index=index,
    llm_client=openai_client
)

response = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [11]:
# Q3. RAG
response.usage.input_tokens

7146

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
# Q4. Chunking
len(chunks)

295

In [14]:
chunked_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunked_index.fit(chunks)

In [15]:
chunked_assistant = RAGLessons(
    index=chunked_index,
    llm_client=openai_client
)

chunked_response = chunked_assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [16]:
# Q5. RAG with chunking
response.usage.input_tokens / chunked_response.usage.input_tokens

3.0682696436238728

In [17]:
# Q6. Turning it into an agent
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

def search(query, num_results=5):
    boost_dict = {'content': 3.0, 'filename': 0.5}
    return chunked_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
